In [ ]:
import openpmd_api as io
import numpy as np
import matplotlib.pyplot as plt
import os

BP5_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"

In [ ]:
series = io.Series(
    BP5_PATH,
    io.Access.read_only, 
    '{"verify_homogeneous_extents": false}'
)

print("Series attributes")
print("=================")

for a in series.attributes:
    print(f"{a:<20} --- {series.get_attribute(a):<20}")

In [ ]:
it = series.snapshots()[0]

print("Iteration attributes")
print("====================")

for a in it.attributes:
    print(f"{a:<10} --- {it.get_attribute(a):<20}")

In [ ]:
subs = it.particles["subsystems"]
subs_ids = subs["id"].load_chunk()
series.flush()
subs_lab = subs.get_attribute('labels')

for l, i in zip(subs_lab, subs_ids):
    print(f"{l:<20} --- {i:>2}")

In [ ]:
times = it.particles["tmp"]["times"][io.Record_Component.SCALAR].load_chunk()

Tritium = it.particles["Tritium"]
T_ss = Tritium["mass_steady"][io.Record_Component.SCALAR]
T_ss_data = T_ss.load_chunk()

T_ts = Tritium["mass"][io.Record_Component.SCALAR]
T_ts_data = T_ts.load_chunk()

series.flush()

In [ ]:
for a in Tritium.attributes:
    print(f"{a:<15} --- {Tritium.get_attribute(a)}")

In [ ]:
import matplotlib.pyplot as plt

fig,ax = plt.subplots(ncols=2, nrows=1,figsize=(10,5))

ax[0].barh(subs_lab, T_ss_data[:], label="T")
ax[0].set_xlabel('mass [g]')
ax[0].set_title('T steady state')

name = 'Water_Detrit'
sindex = subs_lab.index(name)
ax[1].plot(times, T_ts_data[sindex, :], label=name)
ax[1].set_ylabel('time [d]')
ax[1].set_ylabel('mass [g]')
ax[1].set_title('T time series')

plt.show()